# Olist PySpark ETL
Clean project notebook containing only the code needed to run and verify the pipeline.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    IntegerType, DoubleType, DecimalType
)

spark = (
    SparkSession.builder
    .appName("OlistETL")
    .master("local[*]")
    .getOrCreate()
)

spark.version


## Schemas


In [ ]:
order_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_status", StringType()),
    StructField("order_purchase_timestamp", TimestampType()),
    StructField("order_approved_at", TimestampType()),
    StructField("order_delivered_carrier_date", TimestampType()),
    StructField("order_delivered_customer_date", TimestampType()),
    StructField("order_estimated_delivery_date", TimestampType())
])

order_item_schema = StructType([
    StructField("order_id", StringType()),
    StructField("order_item_id", IntegerType()),
    StructField("product_id", StringType()),
    StructField("seller_id", StringType()),
    StructField("shipping_limit_date", TimestampType()),
    StructField("price", DoubleType()),
    StructField("freight_value", DoubleType())
])

customer_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("customer_unique_id", StringType()),
    StructField("customer_zip_code_prefix", StringType()),
    StructField("customer_city", StringType()),
    StructField("customer_state", StringType())
])

product_schema = StructType([
    StructField("product_id", StringType()),
    StructField("product_category_name", StringType()),
    StructField("product_name_lenght", IntegerType()),
    StructField("product_description_lenght", IntegerType()),
    StructField("product_photos_qty", IntegerType()),
    StructField("product_weight_g", IntegerType()),
    StructField("product_length_cm", IntegerType()),
    StructField("product_height_cm", IntegerType()),
    StructField("product_width_cm", IntegerType())
])

seller_schema = StructType([
    StructField("seller_id", StringType()),
    StructField("seller_zip_code_prefix", StringType()),
    StructField("seller_city", StringType()),
    StructField("seller_state", StringType())
])

payment_schema = StructType([
    StructField("order_id", StringType()),
    StructField("payment_sequential", IntegerType()),
    StructField("payment_type", StringType()),
    StructField("payment_installments", IntegerType()),
    StructField("payment_value", DecimalType(18, 2))
])

review_schema = StructType([
    StructField("review_id", StringType()),
    StructField("order_id", StringType()),
    StructField("review_score", IntegerType()),
    StructField("review_comment_title", StringType()),
    StructField("review_comment_message", StringType()),
    StructField("review_creation_date", TimestampType()),
    StructField("review_answer_timestamp", TimestampType())
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType()),
    StructField("geolocation_lat", DoubleType()),
    StructField("geolocation_lng", DoubleType()),
    StructField("geolocation_city", StringType()),
    StructField("geolocation_state", StringType())
])

category_translation_schema = StructType([
    StructField("product_category_name", StringType()),
    StructField("product_category_name_english", StringType())
])


## Project modules


In [ ]:
import sys
import logging

sys.path.append("../src")

from ingestion import ingest_olist_data
from validation import (
    null_checks, duplicate_checks, date_validation,
    valid_values_check, referential_integrity_check
)
from transformation import add_delivery_days, add_date_parts, add_late_delivery_flag
from curation import aggregate_order_items, aggregate_payment, build_order_curated,builder_order_items_curated
from writer import write_parquet


## Ingestion configuration


In [ ]:
olist_config = {
    "orders": {"filename": "olist_orders_dataset.csv", "schema": order_schema},
    "customers": {"filename": "olist_customers_dataset.csv", "schema": customer_schema},
    "geolocation": {"filename": "olist_geolocation_dataset.csv", "schema": geolocation_schema},
    "order_items": {"filename": "olist_order_items_dataset.csv", "schema": order_item_schema},
    "order_payment": {"filename": "olist_order_payments_dataset.csv", "schema": payment_schema},
    "orders_reviews": {"filename": "olist_order_reviews_dataset.csv", "schema": review_schema},
    "products": {"filename": "olist_products_dataset.csv", "schema": product_schema},
    "sellers": {"filename": "olist_sellers_dataset.csv", "schema": seller_schema},
    "category_name_translation": {
        "filename": "product_category_name_translation.csv",
        "schema": category_translation_schema
    }
}


## Ingest all datasets


In [ ]:
logging.basicConfig(level=logging.INFO)

dataframe = {}

for name, config in olist_config.items():
    try:
        dataframe[name] = ingest_olist_data(
            spark=spark,
            raw_path_to_file=config["filename"],
            file_schema=config["schema"]
        )
    except Exception as e:
        print(f"Failed to ingest {config['filename']}: {e}")

for name, df in dataframe.items():
    print(f"{name}: {df.count()}")


## Validate orders


In [ ]:
valid_status = [
    "shipped", "canceled", "invoiced", "created",
    "delivered", "unavailable", "processing", "approved"
]

null_checks(dataframe["orders"]).show()

print("Duplicate order IDs:", duplicate_checks(dataframe["orders"], "order_id").count())

print(
    "Invalid delivery sequence:",
    date_validation(
        dataframe["orders"],
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ).count()
)

print(
    "Invalid order statuses:",
    valid_values_check(dataframe["orders"], "order_status", valid_status).count()
)

print(
    "Orders with missing customers:",
    referential_integrity_check(
        dataframe["orders"],
        dataframe["customers"],
        "customer_id"
    ).count()
)

print(
    "Order items with missing orders:",
    referential_integrity_check(
        dataframe["order_items"],
        dataframe["orders"],
        "order_id"
    ).count()
)


## Transform orders


In [ ]:
orders_transformed = add_delivery_days(dataframe["orders"])
orders_transformed = add_date_parts(
    orders_transformed,
    "order_purchase_timestamp",
    "purchase"
)
orders_transformed = add_late_delivery_flag(orders_transformed)

orders_transformed.select(
    "order_id",
    "delivery_days",
    "purchase_year",
    "purchase_month",
    "is_late"
).show(5)


## Curate order-level dataset


In [ ]:
order_items_agg = aggregate_order_items(dataframe["order_items"])
payment_agg = aggregate_payment(dataframe["order_payment"])

curated_orders = build_order_curated(
    orders_transformed,
    order_items_agg,
    payment_agg,
    dataframe["customers"]
)



## Final checks


In [ ]:
print("Curated order rows:", curated_orders.count())
print("Duplicate order IDs:", duplicate_checks(curated_orders, "order_id").count())

curated_orders.printSchema()
curated_orders.show(5, truncate=False)


In [ ]:
curated_orders_items = builder_order_items_curated(dataframe["order_items"],products_df=dataframe['products'],sellers_df=dataframe['sellers'])
curated_orders_items.count()


In [ ]:
duplicate_order_items_curated = duplicate_checks(dataframe["order_items"], "order_id", "order_item_id")
duplicate_order_items_curated.count()


In [ ]:
duplicate_checks(
    curated_orders_items,
    "order_id",
    "order_item_id"
).count()

In [ ]:
products_referential_integrity_check=referential_integrity_check(parent_df=dataframe["products"],child_df=curated_orders_items,key_column="product_id")
products_referential_integrity_check.count()

In [ ]:
seller_referential_integrity_check=referential_integrity_check(parent_df=dataframe["sellers"],child_df=curated_orders_items,key_column="seller_id")
seller_referential_integrity_check.count()

In [ ]:
write_parquet(curated_orders,output_location="../data/curated/orders/")

In [ ]:
write_parquet(curated_orders_items,output_location="../data/curated/order_items/")

In [ ]:
orders_parquet = spark.read.format("parquet").load("../data/curated/orders/")
orders_parquet.printSchema()

In [ ]:
order_items_parquet = spark.read.format("parquet").load("../data/curated/order_items/")
order_items_parquet.printSchema()

In [ ]:
order_items_parquet.count()